In [ ]:
import sys, os, shutil, tempfile, warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

NOTEBOOK_DIR = Path().resolve()                    # tests/
SRC_DIR      = NOTEBOOK_DIR.parent / "src"         # src/

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Imports ───────────────────────────────────────────────────────────────────
from dataloaders._forking_sequences import fork_sequences
from dataloaders._ts_dataloader      import DataLoaderFactory, FullSeriesDataset
from dataloaders._ts_sharding        import (
    write_sharded_dataset,
    ShardedTrainDataset,
    ShardedValDataset,
    ShardedTestDataset,
)
from common._base_model import BaseModel
from common.train import train, train_distributed, eval_test
from models.linear import TinyLinearModel

warnings.filterwarnings("ignore")
print("imports OK")

In [ ]:
# ── Synthetic dataframe factory ───────────────────────────────────────────────

def make_df(
    n_series: int = 3,
    n_steps:  int = 500,
    n_hist:   int = 1,       # number of hist exog cols
    seed:     int = 42,
) -> pd.DataFrame:
    """
    Returns a long-format DataFrame with columns:
        unique_id, ds, y, available_mask, hist_0, hist_1, ...
    All series have equal length (n_steps timesteps).
    available_mask is 1 everywhere (no gaps).
    """
    rng   = np.random.default_rng(seed)
    times = pd.date_range("2020-01-01", periods=n_steps, freq="5min")
    rows  = []
    for s in range(n_series):
        y    = rng.standard_normal(n_steps).astype(np.float32)
        hist = {f"hist_{i}": rng.standard_normal(n_steps).astype(np.float32)
                for i in range(n_hist)}
        df_s = pd.DataFrame({"unique_id": f"series_{s}", "ds": times, "y": y,
                              "available_mask": np.ones(n_steps, dtype=np.float32),
                              **hist})
        rows.append(df_s)
    return pd.concat(rows, ignore_index=True)


# ── Config factories ──────────────────────────────────────────────────────────

def make_mcfg(
    context_length:    int  = 64,
    fcd_samples:       int  = 4,
    batch_size:        int  = 2,
    max_steps:         int  = 20,
    val_check_interval:int  = 10,
    mixing_strategy:   str  = "concat",
    normalize:         bool = False,
    checkpoint_dir:    str  = "/tmp/ckpts",
    num_workers:       int  = 0,
):
    return SimpleNamespace(
        context_length         = context_length,
        input_size             = context_length,
        fcd_samples            = fcd_samples,
        batch_size             = batch_size,
        valid_batch_size       = batch_size,
        max_steps              = max_steps,
        val_check_interval     = val_check_interval,
        learning_rate          = 1e-3,
        gradient_clip_val      = 1.0,
        early_stopping_patience= 9999,  # disable for tests
        monitor_metric         = "loss",
        monitor_mode           = "min",
        mixing_strategy        = mixing_strategy,
        drop_last              = False,
        normalize              = normalize,
        batch_mode             = "full_series",
        checkpoint_dir         = checkpoint_dir,
        checkpoint_step        = 99999,  # suppress mid-run saves
        num_workers            = num_workers,
    )


def make_entry(
    path:            str,
    name:            str  = "ds",
    horizon:         int  = 6,
    val_size:        int  = 50,
    test_size:       int  = 50,
    weight:          float= 1.0,
    hist_exog_cols:  list = None,
    futr_exog_cols:  list = None,
    stat_exog_cols:  list = None,
    per_series_split:bool = False,
    use_context_head:bool = True,
    sharded_dir:     str  = None,
):
    return SimpleNamespace(
        path             = path,
        name             = name,
        horizon          = horizon,
        val_size         = val_size,
        test_size        = test_size,
        weight           = weight,
        hist_exog_cols   = hist_exog_cols  or [],
        futr_exog_cols   = futr_exog_cols  or [],
        stat_exog_cols   = stat_exog_cols  or [],
        per_series_split = per_series_split,
        use_context_head = use_context_head,
        sharded_dir      = sharded_dir,
    )


def make_dcfg(train_entries, val_entries=None, test_entries=None):
    return SimpleNamespace(
        train      = train_entries,
        validation = val_entries  or train_entries,
        test       = test_entries or train_entries,
    )


print("helpers OK")

In [ ]:
def make_model(mcfg, n_channels=3, n_hist=0):
    return TinyLinearModel(
        context_length = mcfg.context_length,
        horizon        = 6,
        n_channels     = n_channels,
        n_hist         = n_hist,
    )

## Test 3 — `train()`, sharded dataset

In [ ]:
print("=" * 60)
print("TEST 4 — sharded dataset (write → train → val → test)")
print("=" * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    shard_dir = f"{tmpdir}/sharded"
    csv_path  = f"{tmpdir}/data.csv"

    VAL_SIZE  = 60
    TEST_SIZE = 60
    CTX       = 32
    HORIZON   = 6

    df = make_df(n_series=3, n_steps=400, n_hist=1, seed=42)
    df.to_csv(csv_path, index=False)

    # ── Write shards ──────────────────────────────────────────────────────────
    write_sharded_dataset(
        df             = df,
        out_dir        = shard_dir,
        val_size       = VAL_SIZE,
        test_size      = TEST_SIZE,
        context_length = CTX,
        shard_size     = 100,   # small so we get multiple shards
        hist_exog_cols = ["hist_0"],
    )

    shard_files = sorted(Path(shard_dir).glob("shard_*.parquet"))
    print(f"wrote {len(shard_files)} train shard(s) + val.parquet + test.parquet")
    assert (Path(shard_dir) / "val.parquet").exists(),  "val.parquet missing"
    assert (Path(shard_dir) / "test.parquet").exists(), "test.parquet missing"

    # ── Check split sizes ─────────────────────────────────────────────────────
    import json
    meta = json.loads((Path(shard_dir) / "metadata.json").read_text())
    assert meta["T_val"]  == VAL_SIZE,  f"T_val mismatch: {meta['T_val']}"
    assert meta["T_test"] == TEST_SIZE, f"T_test mismatch: {meta['T_test']}"
    T_train_expected = 400 - VAL_SIZE - TEST_SIZE
    assert meta["T_train"] == T_train_expected, f"T_train mismatch: {meta['T_train']}"
    print(f"split sizes: T_train={meta['T_train']}  T_val={meta['T_val']}  T_test={meta['T_test']}")

    # ── Check context heads ───────────────────────────────────────────────────
    val_df  = pd.read_parquet(Path(shard_dir) / "val.parquet")
    test_df = pd.read_parquet(Path(shard_dir) / "test.parquet")
    assert len(val_df)  == VAL_SIZE  + CTX, f"val file length wrong: {len(val_df)}"
    assert len(test_df) == TEST_SIZE + CTX, f"test file length wrong: {len(test_df)}"
    print(f"val.parquet  rows: {len(val_df)}  (= {VAL_SIZE} + {CTX} context)")
    print(f"test.parquet rows: {len(test_df)} (= {TEST_SIZE} + {CTX} context)")

    # ── Train via factory ─────────────────────────────────────────────────────
    mcfg = make_mcfg(
        context_length = CTX,
        fcd_samples    = 4,
        batch_size     = 2,
        max_steps      = 20,
        checkpoint_dir = tmpdir,
    )
    entry = make_entry(
        path           = csv_path,    # still needed for test fallback
        name           = "sharded_ds",
        horizon        = HORIZON,
        val_size       = VAL_SIZE,
        test_size      = TEST_SIZE,
        hist_exog_cols = ["hist_0"],
        sharded_dir    = shard_dir,   # triggers sharded path
    )
    dcfg = make_dcfg([entry])

    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()

    # Verify ShardedTrainDataset is loaded (rank=0, all shards assigned)
    ds = list(train_loader.dataset.datasets)[0]
    assert isinstance(ds, ShardedTrainDataset), f"expected ShardedTrainDataset, got {type(ds)}"
    print(f"ShardedTrainDataset: T={ds.T}  n_windows={ds._n_windows}")

    ds_val = list(factory.val_dataloaders().values())[0].dataset
    assert isinstance(ds_val, ShardedValDataset), f"expected ShardedValDataset, got {type(ds_val)}"
    print(f"ShardedValDataset:   T={ds_val.T}")

    model   = TinyLinearModel(context_length=CTX, horizon=HORIZON, n_channels=3, n_hist=1)
    metrics = train(model, mcfg, train_loader, val_loaders,
                    device=torch.device("cpu"), seed=0)
    print(f"val metrics: {metrics}")

    # Test set uses ShardedTestDataset
    preds = eval_test(model, factory, device=torch.device("cpu"))
    ds_test_loaded = list(factory.test_dataloaders().values())[0].dataset
    assert isinstance(ds_test_loaded, ShardedTestDataset), f"expected ShardedTestDataset"
    print(f"ShardedTestDataset:  T={ds_test_loaded.T}")

print("\n✓ TEST 4 PASSED")

## Test - distributed val sanity check (loss averaged across ranks)

In [ ]:
print("=" * 60)
print("TEST 9 — distributed val: global loss = mean of per-rank losses")
print("         (verified with identical weights AND different weights)")
print("=" * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    shard_dir = f"{tmpdir}/sharded"
    df = make_df(n_series=3, n_steps=400)
    write_sharded_dataset(
        df, shard_dir, val_size=60, test_size=60,
        context_length=32, shard_size=100,
    )

    ds_val = ShardedValDataset(shard_dir, context_length=32, horizon=6)

    from torch.utils.data import DataLoader
    from dataloaders.ts_dataloader import _full_series_collate_fn
    val_loader = DataLoader(ds_val, batch_size=1, collate_fn=_full_series_collate_fn)

    def rank_val_loss(model):
        model.eval()
        total, n = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                fcd_batch = fork_sequences(
                    batch, context_length=32, fcd_samples=-1,
                    horizon=int(batch["horizon"][0].item())
                )
                pred = model(fcd_batch)
                loss = model.loss_fn(
                    pred,
                    fcd_batch["outsample_y"].reshape(-1, 6, 3)
                )
                total += loss.item()
                n     += 1
        return total / n

    def make_model(seed):
        torch.manual_seed(seed)
        m = TinyLinearModel(context_length=32, horizon=6, n_channels=3)
        m.loss_fn = torch.nn.functional.mse_loss
        return m

    for label, seed_r0, seed_r1 in [
        ("identical weights", 0,  0),
        ("different weights", 0, 99),
    ]:
        model_r0 = make_model(seed_r0)
        model_r1 = make_model(seed_r1)

        loss_r0 = rank_val_loss(model_r0)
        loss_r1 = rank_val_loss(model_r1)
        global_loss = (loss_r0 + loss_r1) / 2

        print(f"\n  [{label}]")
        print(f"  rank 0 val loss:  {loss_r0:.6f}")
        print(f"  rank 1 val loss:  {loss_r1:.6f}")
        print(f"  global mean loss: {global_loss:.6f}")

        if seed_r0 == seed_r1:
            assert abs(loss_r0 - loss_r1) < 1e-5, \
                "identical models should get identical val loss"
        else:
            assert abs(loss_r0 - loss_r1) > 1e-5, \
                f"expected different losses for different weights, got {loss_r0:.6f} vs {loss_r1:.6f}"

        assert abs(global_loss - (loss_r0 + loss_r1) / 2) < 1e-9, \
            f"global loss {global_loss:.9f} ≠ mean {(loss_r0 + loss_r1) / 2:.9f}"

print("\n✓ TEST 9 PASSED")